In [1]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory


In [83]:
origem = [1,2]
destino = [4,5]

caminhos = [(1,3),(1,4),(2,3),(2,5),(3,4),(3,5), (5,4)]
custos = [8,15,9,16,6,7,12]

demanda = [20,10]
oferta = [15,15]

In [129]:
model = pyo.ConcreteModel()

model.caminhos = pyo.Set(initialize=caminhos)
model.origens = pyo.Set(initialize=origem)
model.destinos = pyo.Set(initialize=destino)
model.custos = pyo.Param(model.caminhos, initialize=dict(zip(caminhos,custos)))
model.demanda = pyo.Param(model.destinos, initialize=dict(zip(destino,demanda)))
model.oferta = pyo.Param(model.origens, initialize=dict(zip(origem,oferta)))

model.x = pyo.Var(model.caminhos, domain=pyo.NonNegativeIntegers)  




In [130]:
## FUNCÃO OBJETIVO

def funcao_objetivo(model):
    return sum(model.custos[c] * model.x[c] for c in model.caminhos)
model.objetivo = pyo.Objective(rule=funcao_objetivo, sense=pyo.minimize)


In [131]:
## RESTRIÇÕES

def restricao_demanda(model, d):
    entrada = sum(model.x[c] for c in model.caminhos if c[1] == d)
    saida   = sum(model.x[c] for c in model.caminhos if c[0] == d)
    return entrada - saida >= model.demanda[d]

model.restricao_demanda = pyo.Constraint(model.destinos, rule=restricao_demanda)

def restricao_oferta(model, o):
    return sum(model.x[c] for c in model.caminhos if c[0] == o) <= model.oferta[o]
model.restricao_oferta = pyo.Constraint(model.origens, rule=restricao_oferta)


## CAPACIDADE POR ARCO (máximo 10 veículos)
def restricao_capacidade(model, i,j):
    return model.x[i,j] <= 10

model.restricao_capacidade = pyo.Constraint(model.caminhos, rule=restricao_capacidade)

nos_intermediarios = [3]  # não é origem nem destino

def restricao_transbordo(model, n):
    entrada = sum(model.x[c] for c in model.caminhos if c[0] == n)
    saida   = sum(model.x[c] for c in model.caminhos if c[1] == n)
    return entrada == saida

model.restricao_transbordo = pyo.Constraint(nos_intermediarios, rule=restricao_transbordo)


In [132]:
model.pprint()

3 Set Declarations
    caminhos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     2 :    Any :    7 : {(1, 3), (1, 4), (2, 3), (2, 5), (3, 4), (3, 5), (5, 4)}
    destinos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {4, 5}
    origens : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {1, 2}

3 Param Declarations
    custos : Size=7, Index=caminhos, Domain=Any, Default=None, Mutable=False
        Key    : Value
        (1, 3) :     8
        (1, 4) :    15
        (2, 3) :     9
        (2, 5) :    16
        (3, 4) :     6
        (3, 5) :     7
        (5, 4) :    12
    demanda : Size=2, Index=destinos, Domain=Any, Default=None, Mutable=False
        Key : Value
          4 :    20
          5 :    10
    oferta : Size=2, Index=origens, Domain=Any, Default=None, Mutable=Fal

In [133]:
opt = pyo.SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmpmy2b13e3.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpl2csnld9.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpl2csnld9.pyomo.lp
Objective sense      : Minimize
Variables            :       7  [General Integer: 7]
Objective nonzeros   :       7
Linear constraints   :      12  [Less: 9,  Greater: 2,  Equal: 1]
  Nonzeros           :      21
  RHS nonzeros       :      11

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   :

In [134]:
for i in model.caminhos:
    print(f"x[{i}] = {model.x[i].value},custo total = {model.x[i].value * model.custos[i]}    ")

print(f"Custo total = {pyo.value(model.objetivo)}")

x[(1, 3)] = 5.0,custo total = 40.0    
x[(1, 4)] = 10.0,custo total = 150.0    
x[(2, 3)] = 10.0,custo total = 90.0    
x[(2, 5)] = 5.0,custo total = 80.0    
x[(3, 4)] = 10.0,custo total = 60.0    
x[(3, 5)] = 5.0,custo total = 35.0    
x[(5, 4)] = 0.0,custo total = 0.0    
Custo total = 455.0
